# Feature Exploration: High Wind Days (Multi-Route)

This notebook explores the addition of high wind days as a feature. This model builds on the one in notebook 6d, where multi-route with low-volume filtering was implemented. This feature has been explored in the single city-pair case in notebook 4f, where the number of high wind days showed minimal impact on predicting the SYD-MEL pair routes. The motivation here is to test if the feature has more impact on a bigger multi-route data.

Here, the feature is defined as:

`days_high_wind_total` = days_high_wind_dep + days_high_wind_arr

As with the previous feature explorations, the exponential transformation, `days_high_wind_total_exp`, will also be tested.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, f1_score, roc_auc_score
)
import warnings
warnings.filterwarnings('ignore')

# Plotting style
plt.rcParams['text.usetex'] = False
plt.rcParams['font.family'] = 'serif'
plt.rcParams['axes.linewidth'] = 1.5

# XGBoost for classification
try:
    import xgboost as xgb
    HAS_XGB = True
    print("XGBoost available")
except ImportError:
    HAS_XGB = False
    print("XGBoost not installed")

%matplotlib inline

## 1. Data Preparation (same as 6d)

In [ ]:
# Load multi-route data
df = pd.read_csv('../data/processed/ml_training_data_multiroute.csv')

# Parse dates
df['year_month_dt'] = pd.to_datetime(df['year_month'])
df['month_num'] = df['year_month_dt'].dt.month
df['year'] = df['year'].astype(int)

# Create unique identifier for each airline-route combination
df['airline_route'] = df['airline'] + '_' + df['departing_port'] + '_' + df['arriving_port']
df['route'] = df['departing_port'] + '_' + df['arriving_port']

# Sort for proper lag creation
df = df.sort_values(['airline_route', 'year_month_dt']).reset_index(drop=True)

print(f"Original shape: {df.shape}")
print(f"Date range: {df['year_month'].min()} to {df['year_month'].max()}")

## 2. Filter Low-Volume Airline-Routes (from 6d)

In [ ]:
# Calculate average volume per airline-route
volume_threshold = 50  # From notebook 6b

airline_route_volume = df.groupby('airline_route')['sectors_scheduled'].mean().reset_index()
airline_route_volume.columns = ['airline_route', 'avg_volume']

# Identify high-volume airline-routes
high_volume_ar = airline_route_volume[airline_route_volume['avg_volume'] >= volume_threshold]['airline_route'].tolist()

# Filter to high-volume airline-routes only
df_filtered = df[df['airline_route'].isin(high_volume_ar)].copy()

print(f"Volume threshold: {volume_threshold} flights/month")
print(f"\nRecords before filtering: {len(df)}")
print(f"Records after filtering:  {len(df_filtered)}")
print(f"Remaining airline-routes: {df_filtered['airline_route'].nunique()}")

## 3. Feature Engineering

In [ ]:
# Create lagged features
df_filtered['delay_rate_lag1'] = df_filtered.groupby('airline_route')['delay_rate'].shift(1)
df_filtered['delay_rate_lag2'] = df_filtered.groupby('airline_route')['delay_rate'].shift(2)

# Create momentum feature
df_filtered['delay_rate_gradient'] = df_filtered['delay_rate_lag1'] - df_filtered['delay_rate_lag2']

# Create cyclical month encoding
df_filtered['month_sin'] = np.sin(2 * np.pi * df_filtered['month_num'] / 12)
df_filtered['month_cos'] = np.cos(2 * np.pi * df_filtered['month_num'] / 12)

# One-hot encode airline
airline_dummies = pd.get_dummies(df_filtered['airline'], prefix='airline')
df_filtered = pd.concat([df_filtered, airline_dummies], axis=1)
airline_cols = list(airline_dummies.columns)

# One-hot encode route
route_dummies = pd.get_dummies(df_filtered['route'], prefix='route')
df_filtered = pd.concat([df_filtered, route_dummies], axis=1)
route_cols = list(route_dummies.columns)

# Create exponential transformation of rainy_days_arr
df_filtered['rainy_days_arr_exp'] = np.exp(df_filtered['rainy_days_arr'] / df_filtered['rainy_days_arr'].max())

# Create temperature volatility total
df_filtered['temp_volatility_total'] = df_filtered['temp_volatility_dep'] + df_filtered['temp_volatility_arr']
df_filtered['temp_volatility_total_exp'] = np.exp(df_filtered['temp_volatility_total'] / df_filtered['temp_volatility_total'].max())

# NEW: Create high wind days total (dep + arr)
df_filtered['days_high_wind_total'] = df_filtered['days_high_wind_dep'] + df_filtered['days_high_wind_arr']
df_filtered['days_high_wind_total_exp'] = np.exp(df_filtered['days_high_wind_total'] / df_filtered['days_high_wind_total'].max())

print(f"days_high_wind_total range: {df_filtered['days_high_wind_total'].min():.2f} - {df_filtered['days_high_wind_total'].max():.2f}")
print(f"days_high_wind_total_exp range: {df_filtered['days_high_wind_total_exp'].min():.2f} - {df_filtered['days_high_wind_total_exp'].max():.2f}")

# Drop rows with missing lag values
df_clean = df_filtered.dropna(subset=['delay_rate_lag1', 'delay_rate_lag2', 'delay_rate_gradient']).copy()
print(f"\nRows after dropping NaN: {len(df_clean)}")

## 4. Explore Wind Feature Distribution by Route

In [ ]:
# Check wind feature statistics by route
print("High Wind Days Statistics by Route:")
print("=" * 70)
print(f"{'Route':<25} {'Mean':>10} {'Std':>10} {'Max':>10} {'Corr w/delay':>15}")
print("-" * 70)

for route in sorted(df_clean['route'].unique()):
    route_data = df_clean[df_clean['route'] == route]
    mean_wind = route_data['days_high_wind_total'].mean()
    std_wind = route_data['days_high_wind_total'].std()
    max_wind = route_data['days_high_wind_total'].max()
    corr = route_data['delay_rate'].corr(route_data['days_high_wind_total'])
    print(f"{route:<25} {mean_wind:>10.2f} {std_wind:>10.2f} {max_wind:>10.0f} {corr:>15.4f}")

print("\nOverall correlation with delay_rate:")
print(f"  days_high_wind_total: {df_clean['delay_rate'].corr(df_clean['days_high_wind_total']):.4f}")
print(f"  days_high_wind_total_exp: {df_clean['delay_rate'].corr(df_clean['days_high_wind_total_exp']):.4f}")

In [ ]:
# Visualize wind feature distribution by route
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Boxplot of high wind days by route
ax = axes[0]
routes = sorted(df_clean['route'].unique())
wind_by_route = [df_clean[df_clean['route'] == r]['days_high_wind_total'].values for r in routes]
ax.boxplot(wind_by_route, labels=[r.replace('_', '→') for r in routes])
ax.set_ylabel('Days High Wind (Total)')
ax.set_title('High Wind Days Distribution by Route')
ax.tick_params(axis='x', rotation=45)
ax.grid(True, alpha=0.3, axis='y')

# Plot 2: Scatter of wind vs delay_rate by route
ax = axes[1]
colors = plt.cm.tab10(np.linspace(0, 1, len(routes)))
for i, route in enumerate(routes):
    route_data = df_clean[df_clean['route'] == route]
    ax.scatter(route_data['days_high_wind_total'], route_data['delay_rate'], 
               alpha=0.3, s=20, c=[colors[i]], label=route.replace('_', '→'))
ax.set_xlabel('Days High Wind (Total)')
ax.set_ylabel('Delay Rate')
ax.set_title('High Wind Days vs Delay Rate by Route')
ax.legend(fontsize=8, loc='upper right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Train/Validation/Test Split

In [ ]:
# Time-based split (excluding 2020-2022 COVID period)
train_mask = ((df_clean['year'] <= 2017) | (df_clean['year'] == 2024))
val_mask = ((df_clean['year'] == 2018) | (df_clean['year'] == 2023))
test_mask = ((df_clean['year'] == 2019) | (df_clean['year'] >= 2025))

print(f"Train (2010-2017, 2024): {train_mask.sum()} samples")
print(f"Validation (2018, 2023): {val_mask.sum()} samples")
print(f"Test (2019, 2025):       {test_mask.sum()} samples")

In [ ]:
# Define feature sets
base_features = airline_cols + route_cols + ['month_sin', 'month_cos', 'delay_rate_lag1', 'sectors_scheduled']

# Regression features: baseline (6d) vs +wind variants
reg_features_baseline = base_features + ['rainy_days_arr_exp', 'delay_rate_gradient', 'temp_volatility_total_exp']
reg_features_plain = reg_features_baseline + ['days_high_wind_total']
reg_features_exp = reg_features_baseline + ['days_high_wind_total_exp']

# Classification features: baseline (6d) vs +wind variants
clf_features_baseline = base_features + ['rainy_days_arr', 'delay_rate_gradient', 'temp_volatility_total_exp']
clf_features_plain = clf_features_baseline + ['days_high_wind_total']
clf_features_exp = clf_features_baseline + ['days_high_wind_total_exp']

print("Feature sets:")
print(f"  Baseline (6d):        {len(reg_features_baseline)} features")
print(f"  + plain high_wind:    {len(reg_features_plain)} features")
print(f"  + exp high_wind:      {len(reg_features_exp)} features")

In [ ]:
# Prepare target variables
y_train_reg = df_clean.loc[train_mask, 'delay_rate'].values
y_val_reg = df_clean.loc[val_mask, 'delay_rate'].values
y_test_reg = df_clean.loc[test_mask, 'delay_rate'].values

y_train_clf = df_clean.loc[train_mask, 'is_high_delay'].values
y_val_clf = df_clean.loc[val_mask, 'is_high_delay'].values
y_test_clf = df_clean.loc[test_mask, 'is_high_delay'].values

print("Target variables prepared.")

## 6. Regression Models: Baseline vs +Wind

In [ ]:
# Train regression models: baseline vs plain vs exp
reg_results = []

feature_variants = [
    ('baseline', reg_features_baseline),
    ('plain_wind', reg_features_plain),
    ('exp_wind', reg_features_exp)
]

for variant_name, features in feature_variants:
    print(f"\n{'='*60}")
    print(f"Regression with: {variant_name}")
    print(f"{'='*60}")
    
    # Prepare data
    X_train = df_clean.loc[train_mask, features].values
    X_val = df_clean.loc[val_mask, features].values
    X_test = df_clean.loc[test_mask, features].values
    
    # Scale for linear models
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)
    
    # Ridge Regression
    ridge = Ridge(alpha=1.0)
    ridge.fit(X_train_scaled, y_train_reg)
    ridge_test_pred = ridge.predict(X_test_scaled)
    
    ridge_r2 = r2_score(y_test_reg, ridge_test_pred)
    ridge_rmse = np.sqrt(mean_squared_error(y_test_reg, ridge_test_pred))
    print(f"  Ridge:    R²={ridge_r2:.4f}, RMSE={ridge_rmse:.4f}")
    
    reg_results.append({
        'Variant': variant_name,
        'Model': 'Ridge',
        'Test_R2': ridge_r2,
        'Test_RMSE': ridge_rmse
    })
    
    # Random Forest Regression
    rf_reg = RandomForestRegressor(n_estimators=100, max_depth=10, min_samples_leaf=5, random_state=42, n_jobs=-1)
    rf_reg.fit(X_train, y_train_reg)
    rf_test_pred = rf_reg.predict(X_test)
    
    rf_r2 = r2_score(y_test_reg, rf_test_pred)
    rf_rmse = np.sqrt(mean_squared_error(y_test_reg, rf_test_pred))
    print(f"  RF:       R²={rf_r2:.4f}, RMSE={rf_rmse:.4f}")
    
    reg_results.append({
        'Variant': variant_name,
        'Model': 'Random Forest',
        'Test_R2': rf_r2,
        'Test_RMSE': rf_rmse
    })

reg_df = pd.DataFrame(reg_results)

## 7. Classification Models: Baseline vs +Wind

In [ ]:
# Train classification models: baseline vs plain vs exp
clf_results = []

feature_variants = [
    ('baseline', clf_features_baseline),
    ('plain_wind', clf_features_plain),
    ('exp_wind', clf_features_exp)
]

for variant_name, features in feature_variants:
    print(f"\n{'='*60}")
    print(f"Classification with: {variant_name}")
    print(f"{'='*60}")
    
    # Prepare data
    X_train = df_clean.loc[train_mask, features].values
    X_val = df_clean.loc[val_mask, features].values
    X_test = df_clean.loc[test_mask, features].values
    
    # Scale for linear models
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)
    
    # Logistic Regression
    logreg = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
    logreg.fit(X_train_scaled, y_train_clf)
    logreg_test_pred = logreg.predict(X_test_scaled)
    logreg_test_proba = logreg.predict_proba(X_test_scaled)[:, 1]
    
    logreg_f1 = f1_score(y_test_clf, logreg_test_pred)
    logreg_auc = roc_auc_score(y_test_clf, logreg_test_proba)
    print(f"  Logistic: F1={logreg_f1:.4f}, AUC={logreg_auc:.4f}")
    
    clf_results.append({
        'Variant': variant_name,
        'Model': 'Logistic',
        'Test_F1': logreg_f1,
        'Test_AUC': logreg_auc
    })
    
    # Random Forest Classification
    rf_clf = RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_leaf=5, random_state=42, n_jobs=-1)
    rf_clf.fit(X_train, y_train_clf)
    rf_clf_test_pred = rf_clf.predict(X_test)
    rf_clf_test_proba = rf_clf.predict_proba(X_test)[:, 1]
    
    rf_clf_f1 = f1_score(y_test_clf, rf_clf_test_pred)
    rf_clf_auc = roc_auc_score(y_test_clf, rf_clf_test_proba)
    print(f"  RF Clf:   F1={rf_clf_f1:.4f}, AUC={rf_clf_auc:.4f}")
    
    clf_results.append({
        'Variant': variant_name,
        'Model': 'Random Forest',
        'Test_F1': rf_clf_f1,
        'Test_AUC': rf_clf_auc
    })
    
    # XGBoost Classification
    if HAS_XGB:
        xgb_clf = xgb.XGBClassifier(
            n_estimators=100, max_depth=5, learning_rate=0.1,
            min_child_weight=5, random_state=42, n_jobs=-1
        )
        xgb_clf.fit(X_train, y_train_clf, eval_set=[(X_val, y_val_clf)], verbose=False)
        xgb_test_pred = xgb_clf.predict(X_test)
        xgb_test_proba = xgb_clf.predict_proba(X_test)[:, 1]
        
        xgb_f1 = f1_score(y_test_clf, xgb_test_pred)
        xgb_auc = roc_auc_score(y_test_clf, xgb_test_proba)
        print(f"  XGBoost:  F1={xgb_f1:.4f}, AUC={xgb_auc:.4f}")
        
        clf_results.append({
            'Variant': variant_name,
            'Model': 'XGBoost',
            'Test_F1': xgb_f1,
            'Test_AUC': xgb_auc
        })

clf_df = pd.DataFrame(clf_results)

## 8. Side-by-Side Comparison

In [ ]:
# Regression comparison
print("=" * 100)
print("REGRESSION: baseline (6d) vs +plain_wind vs +exp_wind")
print("=" * 100)

reg_pivot = reg_df.pivot(index='Model', columns='Variant', values=['Test_R2', 'Test_RMSE'])
reg_pivot.columns = [f'{col[1]}_{col[0]}' for col in reg_pivot.columns]
reg_pivot = reg_pivot.reset_index()

# Calculate differences from baseline
reg_pivot['plain_R2_diff'] = reg_pivot['plain_wind_Test_R2'] - reg_pivot['baseline_Test_R2']
reg_pivot['exp_R2_diff'] = reg_pivot['exp_wind_Test_R2'] - reg_pivot['baseline_Test_R2']

print(f"\n{'Model':<15} {'baseline R²':>12} {'plain R²':>12} {'exp R²':>12} {'plain Δ':>10} {'exp Δ':>10}")
print("-" * 80)
for _, row in reg_pivot.iterrows():
    plain_sign = '+' if row['plain_R2_diff'] > 0 else ''
    exp_sign = '+' if row['exp_R2_diff'] > 0 else ''
    print(f"{row['Model']:<15} {row['baseline_Test_R2']:>12.4f} {row['plain_wind_Test_R2']:>12.4f} {row['exp_wind_Test_R2']:>12.4f} {plain_sign}{row['plain_R2_diff']:>9.4f} {exp_sign}{row['exp_R2_diff']:>9.4f}")

In [ ]:
# Classification comparison
print("=" * 100)
print("CLASSIFICATION: baseline (6d) vs +plain_wind vs +exp_wind")
print("=" * 100)

clf_pivot = clf_df.pivot(index='Model', columns='Variant', values=['Test_F1', 'Test_AUC'])
clf_pivot.columns = [f'{col[1]}_{col[0]}' for col in clf_pivot.columns]
clf_pivot = clf_pivot.reset_index()

# Calculate differences from baseline
clf_pivot['plain_F1_diff'] = clf_pivot['plain_wind_Test_F1'] - clf_pivot['baseline_Test_F1']
clf_pivot['exp_F1_diff'] = clf_pivot['exp_wind_Test_F1'] - clf_pivot['baseline_Test_F1']

print(f"\n{'Model':<15} {'baseline F1':>12} {'plain F1':>12} {'exp F1':>12} {'plain Δ':>10} {'exp Δ':>10}")
print("-" * 80)
for _, row in clf_pivot.iterrows():
    plain_sign = '+' if row['plain_F1_diff'] > 0 else ''
    exp_sign = '+' if row['exp_F1_diff'] > 0 else ''
    print(f"{row['Model']:<15} {row['baseline_Test_F1']:>12.4f} {row['plain_wind_Test_F1']:>12.4f} {row['exp_wind_Test_F1']:>12.4f} {plain_sign}{row['plain_F1_diff']:>9.4f} {exp_sign}{row['exp_F1_diff']:>9.4f}")

In [ ]:
# Visualization: Side-by-side bar charts
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Regression R²
ax = axes[0]
x = np.arange(len(reg_pivot))
width = 0.25
ax.bar(x - width, reg_pivot['baseline_Test_R2'], width, label='baseline (6d)', color='steelblue', alpha=0.8)
ax.bar(x, reg_pivot['plain_wind_Test_R2'], width, label='+ plain wind', color='#27ae60', alpha=0.8)
ax.bar(x + width, reg_pivot['exp_wind_Test_R2'], width, label='+ exp wind', color='#e67e22', alpha=0.8)
ax.set_ylabel(r'Test $R^2$')
ax.set_title(r'Regression: $R^2$ Comparison')
ax.set_xticks(x)
ax.set_xticklabels(reg_pivot['Model'])
ax.legend()
ax.set_ylim(0, 0.6)
ax.grid(True, alpha=0.3, axis='y')

# Classification F1
ax = axes[1]
x = np.arange(len(clf_pivot))
ax.bar(x - width, clf_pivot['baseline_Test_F1'], width, label='baseline (6d)', color='steelblue', alpha=0.8)
ax.bar(x, clf_pivot['plain_wind_Test_F1'], width, label='+ plain wind', color='#27ae60', alpha=0.8)
ax.bar(x + width, clf_pivot['exp_wind_Test_F1'], width, label='+ exp wind', color='#e67e22', alpha=0.8)
ax.set_ylabel(r'Test $F_1$')
ax.set_title(r'Classification: $F_1$ Comparison')
ax.set_xticks(x)
ax.set_xticklabels(clf_pivot['Model'])
ax.legend()
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 9. Feature Importance Analysis

In [ ]:
# Train RF with exp wind to check feature importance
X_train_reg = df_clean.loc[train_mask, reg_features_exp].values
rf_final = RandomForestRegressor(n_estimators=100, max_depth=10, min_samples_leaf=5, random_state=42, n_jobs=-1)
rf_final.fit(X_train_reg, y_train_reg)

# Feature importance
importance_df = pd.DataFrame({
    'Feature': reg_features_exp,
    'Importance': rf_final.feature_importances_
}).sort_values('Importance', ascending=False)

print("Feature Importance (RF Regression with exp wind):")
print("-" * 50)
for _, row in importance_df.head(15).iterrows():
    marker = " ← NEW" if 'wind' in row['Feature'] else ""
    print(f"  {row['Feature']:<30} {row['Importance']:.4f}{marker}")

# Highlight wind feature
wind_importance = importance_df[importance_df['Feature'] == 'days_high_wind_total_exp']['Importance'].values[0]
wind_rank = list(importance_df['Feature']).index('days_high_wind_total_exp') + 1
print(f"\ndays_high_wind_total_exp: rank {wind_rank}, importance {wind_importance:.4f}")

In [ ]:
# Visualize top features
fig, ax = plt.subplots(figsize=(10, 8))

top_features = importance_df.head(15)

def get_feature_color(f):
    if 'wind' in f:
        return '#e67e22'  # orange for wind (NEW)
    elif 'route_' in f:
        return '#e74c3c'  # red for route
    elif 'airline_' in f:
        return '#9b59b6'  # purple for airline
    elif 'temp_volatility' in f:
        return '#f39c12'  # yellow for temp_volatility
    elif 'rainy' in f:
        return '#3498db'  # blue for rain
    elif 'gradient' in f:
        return '#27ae60'  # green for momentum
    elif 'lag' in f:
        return '#1abc9c'  # teal for lag
    else:
        return 'steelblue'

colors = [get_feature_color(f) for f in top_features['Feature']]

ax.barh(top_features['Feature'], top_features['Importance'], color=colors, alpha=0.8)
ax.set_xlabel('Feature Importance')
ax.set_title('Top 15 Feature Importances (RF Regression)\n(orange=wind, red=route, purple=airline, green=momentum)')
ax.invert_yaxis()

plt.tight_layout()
plt.show()

## 10. Comparison with 4f (Single-Route) and 6d (Baseline)

In [ ]:
# Reference values from 6d (baseline multi-route filtered)
ref_6d = {
    'Ridge': {'R2': 0.4487},
    'Random Forest': {'R2': 0.4605},
    'Logistic': {'F1': 0.7523},
    'Random Forest_clf': {'F1': 0.7436},
    'XGBoost': {'F1': 0.7421}
}

# Reference from 4f: wind feature had minimal impact on single-route
# (checking if multi-route is different)

print("="*80)
print("COMPARISON: 6d (baseline) vs 6e (with wind feature)")
print("="*80)

print("\nREGRESSION (R²):")
print("-" * 70)
print(f"{'Model':<20} {'6d baseline':>12} {'6e best':>12} {'Δ':>10} {'Best variant':>15}")
print("-" * 70)

for _, row in reg_pivot.iterrows():
    model = row['Model']
    baseline = row['baseline_Test_R2']
    plain = row['plain_wind_Test_R2']
    exp = row['exp_wind_Test_R2']
    
    best_r2 = max(plain, exp)
    best_variant = 'exp' if exp >= plain else 'plain'
    diff = best_r2 - baseline
    sign = '+' if diff > 0 else ''
    
    status = 'IMPROVED' if diff > 0.005 else ('WORSE' if diff < -0.005 else '~same')
    print(f"{model:<20} {baseline:>12.4f} {best_r2:>12.4f} {sign}{diff:>9.4f} {best_variant:>15} ({status})")

print("\nCLASSIFICATION (F1):")
print("-" * 70)
print(f"{'Model':<20} {'6d baseline':>12} {'6e best':>12} {'Δ':>10} {'Best variant':>15}")
print("-" * 70)

for _, row in clf_pivot.iterrows():
    model = row['Model']
    baseline = row['baseline_Test_F1']
    plain = row['plain_wind_Test_F1']
    exp = row['exp_wind_Test_F1']
    
    best_f1 = max(plain, exp)
    best_variant = 'exp' if exp >= plain else 'plain'
    diff = best_f1 - baseline
    sign = '+' if diff > 0 else ''
    
    status = 'IMPROVED' if diff > 0.005 else ('WORSE' if diff < -0.005 else '~same')
    print(f"{model:<20} {baseline:>12.4f} {best_f1:>12.4f} {sign}{diff:>9.4f} {best_variant:>15} ({status})")

## 11. Summary and Observations

In [ ]:
print("="*80)
print("SUMMARY: Impact of High Wind Days on Multi-Route Model")
print("="*80)

# Summary metrics table - Regression
print("\nREGRESSION PERFORMANCE:")
print("-" * 70)
print(f"{'Model':<20} {'6d Baseline':>12} {'6e +Wind':>12} {'Δ R²':>10} {'Status':>12}")
print("-" * 70)
for _, row in reg_pivot.iterrows():
    model = row['Model']
    baseline = row['baseline_Test_R2']
    best = max(row['plain_wind_Test_R2'], row['exp_wind_Test_R2'])
    diff = best - baseline
    sign = '+' if diff > 0 else ''
    status = 'IMPROVED' if diff > 0.005 else ('WORSE' if diff < -0.005 else '~same')
    print(f"{model:<20} {baseline:>12.4f} {best:>12.4f} {sign}{diff:>9.4f} {status:>12}")

# Summary metrics table - Classification
print("\nCLASSIFICATION PERFORMANCE:")
print("-" * 70)
print(f"{'Model':<20} {'6d Baseline':>12} {'6e +Wind':>12} {'Δ F1':>10} {'Status':>12}")
print("-" * 70)
for _, row in clf_pivot.iterrows():
    model = row['Model']
    baseline = row['baseline_Test_F1']
    best = max(row['plain_wind_Test_F1'], row['exp_wind_Test_F1'])
    diff = best - baseline
    sign = '+' if diff > 0 else ''
    status = 'IMPROVED' if diff > 0.005 else ('WORSE' if diff < -0.005 else '~same')
    print(f"{model:<20} {baseline:>12.4f} {best:>12.4f} {sign}{diff:>9.4f} {status:>12}")

# Count improvements
reg_improvements = ((reg_pivot['plain_R2_diff'] > 0.005) | (reg_pivot['exp_R2_diff'] > 0.005)).sum()
clf_improvements = ((clf_pivot['plain_F1_diff'] > 0.005) | (clf_pivot['exp_F1_diff'] > 0.005)).sum()

total_models = len(reg_pivot) + len(clf_pivot)
total_improvements = reg_improvements + clf_improvements


### Observations

Adding the exponentially scaled number of windy days, `days_high_wind_total_exp`, is the better variant of the feature. In general, it positively impacts the performance of the prediction models:
- **improved** both regression models: average improvement of +0.0183 R²
- slightly **improved** the classification models: average F1 score improvement of +0.0096, with the exception of the XX model.

Compared with all features used in the prediction models, it is ranked 5 out of 21 features. This is a different outcome from notebook 4f.

## 12. Next step

Motivated by the different outcome from the earlier single-pair routes, the holiday features will be revisited in the next notebook.

See notebook [7a_feature_exploration_holidays_multiroute.ipynb](/notebooks/7a_feature_exploration_holidays_multiroute.ipynb).